# The AG Share of New Company Registrations in Basel-Landschaft, 2017 vs 2025

**Author:** Mauro Reverberi

**Program:** MSc AI, Udacity Institute of AI & Technology / Woolf

**Project:** Capstone Project 2, Data and Statistical Reasoning

**Dataset:** "Firmenmutationen nach Rechtsform und Gemeinde" (company mutations by legal form and municipality), Open Government Data Kanton Basel-Landschaft:
https://data.bl.ch/explore/dataset/12460/

In this notebook I analyze new company registrations in the Swiss canton of
Basel-Landschaft.

**Research question:** Did the share of stock corporations (Aktiengesellschaft,
AG) among new company registrations in the canton of Basel-Landschaft change
between 2017 and 2025?

I picked this question because it connects to my first capstone project. In
Project 1 I analyzed the Swiss entries of the global LEI dataset and found
that the classic stock corporation (AG / SA) is the largest legal form group
in that register. This made me curious whether the AG also plays such a
strong role when new companies are founded, and whether that role changed
over time. One important point up front, the two projects use different
datasets and different populations. Project 1 looked at the stock of
LEI-registered entities, which is dominated by larger firms. This project
looks at single new registrations published in the Swiss Official Gazette of
Commerce (SHAB) for one canton. The percentages from the two projects must
not be compared directly.

## Dataset

The frozen snapshot `firmenmutationen.csv` is stored in the same folder as
this notebook. I downloaded it on 2026-08-13 through the direct CSV export:
https://data.bl.ch/explore/dataset/12460/download/?format=csv

The snapshot covers the period 2016-02-03 to 2026-08-12.

One row in this dataset is one single mutation message published in the
SHAB, so the data is not aggregated. The observation unit of my analysis is
one published new company registration. The online dataset is updated
continuously, so I froze a local copy and the notebook only reads this
local file. The publisher documents two known quality issues, the
municipality columns are partly incomplete and the free text field
`meldung` contains encoding errors. Both fields are not needed for my
research question.

## Setup

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

## 1. Load the dataset

I load the frozen CSV with a semicolon separator and check that the structure looks right before I touch anything.

In [7]:
df = pd.read_csv("firmenmutationen.csv", sep=";", encoding="utf-8")
df.head()

,kategorie,publikationsdatum_shab,journaldatum_handelsregister,id_shab,firmensitz_code,firmensitz,meldung,uid,firmenname,rechtsform_code,rechtsform
0,Neueintragung,2026-08-12,2026-08-07,1006729550,2762.0,Allschwil,"R.K. SanitÃ¤r Rima, in Allschwil, CHE-440.128....",CHE440128482,R.K. Sanitär Rima,101,Einzelunternehmen
1,Neueintragung,2026-08-12,2026-08-07,1006729549,2762.0,Allschwil,"R.K. Elektro Rima, in Allschwil, CHE-380.378.8...",CHE380378843,R.K. Elektro Rima,101,Einzelunternehmen
2,Neueintragung,2026-08-12,2026-08-07,1006729548,2834.0,Ziefen,"MS Swiss Handel GmbH, in Ziefen, CHE-285.436.8...",CHE285436803,MS Swiss Handel GmbH,107,Gesellschaft mit beschränkter Haftung (GmbH)
3,Löschung,2026-08-12,2026-08-07,1006729555,2824.0,Frenkendorf,"OlaVintage, Inhaber John Jairo Lopez, in Frenk...",CHE354942212,"OlaVintage, Inhaber John Jairo Lopez",101,Einzelunternehmen
4,Löschung,2026-08-12,2026-08-07,1006729554,2761.0,Aesch (BL),"MB Car Cleaning Jeyson Benitez Gomez, in Aesch...",CHE363228510,MB Car Cleaning Jeyson Benitez Gomez,101,Einzelunternehmen


The first rows look as expected, one register message per row with the mutation category, two dates, the company name, the legal form and a long free text field. The encoding problem the publisher mentions is directly visible, `meldung` shows broken characters like "SanitÃ¤r" while the separate `firmenname` column shows the correct "Sanitär". So the problem is only in the free text field, which I do not use.


In [8]:
print(f"Rows: {df.shape[0]}, columns: {df.shape[1]}")
df.info()

Rows: 28992, columns: 11
<class 'pandas.DataFrame'>
RangeIndex: 28992 entries, 0 to 28991
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   kategorie                     28992 non-null  str    
 1   publikationsdatum_shab        28992 non-null  str    
 2   journaldatum_handelsregister  28992 non-null  str    
 3   id_shab                       28992 non-null  int64  
 4   firmensitz_code               28882 non-null  float64
 5   firmensitz                    28882 non-null  str    
 6   meldung                       28992 non-null  str    
 7   uid                           28992 non-null  str    
 8   firmenname                    28992 non-null  str    
 9   rechtsform_code               28992 non-null  int64  
 10  rechtsform                    28992 non-null  str    
dtypes: float64(1), int64(2), str(8)
memory usage: 2.4 MB


The file has 28,992 rows and 11 columns, which matches the download page. Three columns are numeric (`id_shab`, `firmensitz_code`, `rechtsform_code`), all of them are identifiers or codes. The two date columns are loaded as strings, so I will convert the publication date later. Nothing needs fixing at this point.